# Data Cleaning Phase
This notebook is currently in the data cleaning phase.

**Dataset file used:** `Dataset.xlsx`

Key steps in this phase:
- Load the raw dataset (Dataset.xlsx)
- Inspect for missing and duplicate values
- Standardize categorical values (e.g., country names)
- Convert date columns and save cleaned outputs

Proceed to run the next cell to load the dataset and begin cleaning.

# Setup

Installing Libraries, loading the dataset and capturing basic metadata (rows/cols).

In [1]:
pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import datetime as dt

df = pd.read_excel("Dataset.xlsx", dtype=str, engine='openpyxl')  # dtype=str to prevent auto-conversion

# Initial inspection and quick fixes

Getting head/tail/summary/dtypes, missing-value stats, and flag likely numeric-or-date columns stored as text.

#### Quick inspection
    - head
    - tail
    - summary
    - dtypes


In [3]:
# Quick inspection
cp = df.copy()
print('HEAD:')
cp.head(2)


HEAD:


,Learner SignUp DateTime,Opportunity Id,Opportunity Name,Opportunity Category,Opportunity End Date,First Name,Date of Birth,Gender,Country,Institution Name,Current/Intended Major,Entry created at,Status Description,Status Code,Apply Date,Opportunity Start Date
0,06/14/2023 12:30:35,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,Course,06/29/2024 18:52:39,Faria,2001-12-01 00:00:00,Female,Pakistan,Nwihs,Radiology,2024-11-03 12:01:41,Started,1080,06/14/2023 12:36:09,2022-03-11 18:30:39
1,2023-01-05 05:29:16,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,Course,06/29/2024 18:52:39,Poojitha,08/16/2000,Female,India,SAINT LOUIS,Information Systems,2024-11-03 12:01:41,Started,1080,2023-01-05 06:08:21,2022-03-11 18:30:39


In [4]:
print('\nTAIL:')
cp.tail(2)


TAIL:


,Learner SignUp DateTime,Opportunity Id,Opportunity Name,Opportunity Category,Opportunity End Date,First Name,Date of Birth,Gender,Country,Institution Name,Current/Intended Major,Entry created at,Status Description,Status Code,Apply Date,Opportunity Start Date
8556,12/23/2023 03:53:12,00000000-10GQ-RJHT-3G4S-BKGBY1,Freelance Mastery workshop,Event,2024-08-03 11:30:00,venumadhavi,1997-05-06 00:00:00,Female,United States,Saint Louis University,Information Systems,2024-11-03 12:03:14,Team Allocated,1070,02/27/2024 06:48:47,2024-08-03 14:00:00
8557,2023-01-06 13:22:01,00000000-10GQ-RJHT-3G4S-BKGBY1,Freelance Mastery workshop,Event,2024-08-03 11:30:00,Prashanth Reddy,11/30/2000,Male,India,Saint Louis University,Artificial Intelligence,2024-11-03 12:03:14,Team Allocated,1070,02/16/2024 16:33:00,2024-08-03 14:00:00


In [5]:
# Summary
cp.describe(include='all').T

,count,unique,top,freq
Learner SignUp DateTime,8558,3916,01/31/2024 15:11:57,16
Opportunity Id,8558,23,00000000-0GN2-A0AY-7XK8-C5FZPP,1423
Opportunity Name,8558,22,Career Essentials: Getting Started with Your P...,1423
Opportunity Category,8558,5,Internship,5421
Opportunity End Date,8558,17,2024-11-03 18:00:00,4159
First Name,8558,3026,Bhargavi,40
Date of Birth,8558,2620,05/20/2001,29
Gender,8558,4,Male,5018
Country,8558,71,United States,3976
Institution Name,8553,2089,Saint Louis University,2631


In [6]:
# Dtypes
print("\nDTYPES:")
cp.dtypes


DTYPES:


Learner SignUp DateTime    object
Opportunity Id             object
Opportunity Name           object
Opportunity Category       object
Opportunity End Date       object
First Name                 object
Date of Birth              object
Gender                     object
Country                    object
Institution Name           object
Current/Intended Major     object
Entry created at           object
Status Description         object
Status Code                object
Apply Date                 object
Opportunity Start Date     object
dtype: object

In [7]:
# Missing value counts and percent
miss = cp.isna().sum()
miss_pct = miss / len(cp) * 100
miss_summary = pd.DataFrame({'missing': miss, 'percent': miss_pct})
print('\nMissing value summary (top 10):')
print(miss_summary.sort_values('percent', ascending=False).head(10))


Missing value summary (top 10):
                         missing    percent
Opportunity Start Date      3794  44.332788
Institution Name               5   0.058425
Current/Intended Major         5   0.058425
Learner SignUp DateTime        0   0.000000
Opportunity End Date           0   0.000000
Opportunity Id                 0   0.000000
Opportunity Name               0   0.000000
Opportunity Category           0   0.000000
Gender                         0   0.000000
Date of Birth                  0   0.000000


In [8]:
# Columns with >10% missing
high_missing = miss_summary[miss_summary['percent'] > 10]
if not high_missing.empty:
    print('\nColumns with >10% missing:')
    print(high_missing)


Columns with >10% missing:
                        missing    percent
Opportunity Start Date     3794  44.332788


In [9]:

# Flag columns that look numeric but are object dtype
numeric_like = []
for c in cp.columns:
    if cp[c].dtype == object:
        sample = cp[c].dropna().astype(str).head(20).tolist()
        # if all sample values look like numbers
        if sample and all(s.replace(',', '').replace('.', '').isdigit() for s in sample):
            numeric_like.append(c)

if numeric_like:
    print('\nColumns that look numeric but are text:\n', numeric_like)


Columns that look numeric but are text:
 ['Status Code']


In [10]:

# Flag date-like columns stored as text (simple heuristic)
possible_dates = []
for c in cp.columns:
    if cp[c].dtype == object:
        s = cp[c].dropna().astype(str).head(50).tolist()
        if s:
            # heuristic: many rows contain '/' or '-' and digits
            score = sum(1 for v in s if ('/' in v or '-' in v) and any(ch.isdigit() for ch in v))
            if score > len(s) * 0.5:
                possible_dates.append(c)

if possible_dates:
    print('\nColumns that may be dates stored as text:\n', possible_dates)


Columns that may be dates stored as text:
 ['Learner SignUp DateTime', 'Opportunity Id', 'Opportunity End Date', 'Date of Birth', 'Entry created at', 'Apply Date', 'Opportunity Start Date']


# Converting Date types

In [14]:

# Convert only the true date columns to datetime
date_columns = [
    "Learner SignUp DateTime",
    "Opportunity End Date",
    "Date of Birth",
    "Entry created at",
    "Apply Date",
    "Opportunity Start Date"
]
for col in date_columns:
    cp[col] = pd.to_datetime(cp[col], format='mixed', errors='coerce')

cp["Date of Birth"] = pd.to_datetime(cp["Date of Birth"], format='mixed', errors='coerce').dt.date


print(cp.dtypes)

#inputting nan is some missing valus (handling missing data)
cp['Opportunity Start Date']=cp['Opportunity Start Date'].fillna('unknown')

# Drop null rows and save cleaned file
cp_cleaned = cp.dropna()
cp_cleaned.head()

Learner SignUp DateTime    datetime64[ns]
Opportunity Id             string[python]
Opportunity Name           string[python]
Opportunity Category       string[python]
Opportunity End Date       datetime64[ns]
First Name                 string[python]
Date of Birth                      object
Gender                     string[python]
Country                    string[python]
Institution Name           string[python]
Current/Intended Major     string[python]
Entry created at           datetime64[ns]
Status Description         string[python]
Status Code                         int64
Apply Date                 datetime64[ns]
Opportunity Start Date     datetime64[ns]
dtype: object


,Learner SignUp DateTime,Opportunity Id,Opportunity Name,Opportunity Category,Opportunity End Date,First Name,Date of Birth,Gender,Country,Institution Name,Current/Intended Major,Entry created at,Status Description,Status Code,Apply Date,Opportunity Start Date
0,2023-06-14 12:30:35,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,Course,2024-06-29 18:52:39,Faria,2001-12-01,Female,Pakistan,Nwihs,Radiology,2024-11-03 12:01:41,Started,1080,2023-06-14 12:36:09,2022-03-11 18:30:39
1,2023-01-05 05:29:16,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,Course,2024-06-29 18:52:39,Poojitha,2000-08-16,Female,India,SAINT LOUIS,Information Systems,2024-11-03 12:01:41,Started,1080,2023-01-05 06:08:21,2022-03-11 18:30:39
3,2023-08-29 05:20:03,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,Course,2024-06-29 18:52:39,Amrutha Varshini,1999-01-11,Female,United States,Saint Louis University,Information Systems,2024-11-03 12:01:41,Team Allocated,1070,2023-09-10 22:02:42,2022-03-11 18:30:39
4,2023-06-01 15:26:36,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,Course,2024-06-29 18:52:39,Vinay Varshith,2000-04-19,Male,United States,Saint Louis University,Computer Science,2024-11-03 12:01:41,Started,1080,2023-06-01 15:40:10,2022-03-11 18:30:39
5,2024-02-03 19:16:07,00000000-0GN2-A0AY-7XK8-C5FZPP,Career Essentials: Getting Started with Your P...,Course,2024-06-29 18:52:39,Mor,1996-12-05,Male,India,Saint Louis University,Mechanical Engineering,2024-11-03 12:01:41,Waitlisted,1040,2024-02-03 20:30:35,unknown


In [12]:
# Drop exact duplicate rows
before = cp.shape[0]
cp = cp.drop_duplicates()
after = cp.shape[0]
print(f'Dropped exact duplicates: {before - after}. Current shape: {cp.shape}')

Dropped exact duplicates: 0. Current shape: (8558, 16)


In [13]:
# Drop rows where Institution Name or Current/Intended Major are null/NA
cp = cp.dropna(subset=['Institution Name', 'Current/Intended Major'])
print(f"Rows remaining: {cp.shape[0]}")

Rows remaining: 8548


In [15]:
cp_cleaned.shape

(6839, 16)

# Duplicates and consistency

- Drop exact duplicate rows.
- Identify near-duplicates using key columns: Opportunity Id, Learner SignUp DateTime, First Name. Print a small sample for manual review.

In [16]:
# Duplicates and consistency (idempotent)
# 1) Drop exact duplicates
before = cp_cleaned.shape[0]
cp_cleaned = cp_cleaned.drop_duplicates()
after = cp_cleaned.shape[0]
print(f'Dropped exact duplicates: {before - after}; current shape: {cp_cleaned.shape}')

# 2) Identify near-duplicates by keys
keys = ['Opportunity Id','Learner SignUp DateTime','First Name']
present_keys = [k for k in keys if k in cp.columns]
if len(present_keys) < 2:
    print('Not enough key columns present to identify near-duplicates. Present keys:', present_keys)
else:
    # find rows sharing the same key tuple (but do not create a column)
    dup_mask = cp_cleaned.duplicated(subset=present_keys, keep=False)
    near_dup_count = int(dup_mask.sum())
    print(f'Near-duplicates found (by {present_keys}): {near_dup_count}')
    if near_dup_count:
        sample_cols = present_keys + [c for c in ['Institution Name','Opportunity Name','Status Code'] if c in cp.columns]
        print('\nSample near-duplicates (up to 10):')
        print(cp_cleaned.loc[dup_mask, sample_cols].head(10))

Dropped exact duplicates: 0; current shape: (8548, 16)
Near-duplicates found (by ['Opportunity Id', 'Learner SignUp DateTime', 'First Name']): 0
Flag column 'near_duplicate_by_key' created/updated.


In [ ]:
cp_cleaned.to_csv("cleaned.csv", index=False, encoding="utf-8")

## Compact validation report
Key validation points to check on the cleaned dataset:
- Total rows and columns after primary cleaning (head/tail checks were kept conservative).
- Missingness: `Opportunity Start Date` (high missingness), `Institution Name`, and `Current/Intended Major` (small counts).
- Date parsing: verify `Date of Birth` parse success count and remaining NaT values.
- Duplicates: exact duplicates removed; near-duplicates reported (if any).
Use the code cell that follows to produce the exact numeric outputs for these checks. Paste those outputs here or keep them as an execution trace for the audit.

In [ ]:
# Compact numeric summary for validation (run this cell after running earlier cells)
def compact_validation_report(df):
    rows, cols = df.shape
    miss = df.isna().sum()
    high_missing = miss[miss / rows * 100 > 10] if rows else pd.Series([], dtype=int)
    dob_nat = 0
    if 'Date of Birth' in df.columns:
        dob_nat = int(pd.to_datetime(df['Date of Birth'], errors='coerce').isna().sum())
    exact_dups = int(df.duplicated(keep='first').sum())
    print(f'Records: {rows} rows x {cols} cols')
    print('Top 10 columns by missing values:')
    for c, v in miss.sort_values(ascending=False).head(10).items():
        print(f' - {c}: {v} ({v/rows*100:.2f}%)')
    print('Columns with >10% missing:')
    if not high_missing.empty:
        print(high_missing)
    else:
        print(' - none')
    print(f'Exact duplicate rows remaining: {exact_dups}')

# Run the report on the working DataFrame variables that are present
for var in ['cp_cleaned', 'cp', 'df']:
    if var in globals():
        print('--- summary for', var, '---')
        compact_validation_report(globals()[var])
        print()

--- summary for cp_cleaned ---
Records: 6839 rows x 16 cols
Top 10 columns by missing values:
 - Learner SignUp DateTime: 0 (0.00%)
 - Opportunity Id: 0 (0.00%)
 - Opportunity Name: 0 (0.00%)
 - Opportunity Category: 0 (0.00%)
 - Opportunity End Date: 0 (0.00%)
 - First Name: 0 (0.00%)
 - Date of Birth: 0 (0.00%)
 - Gender: 0 (0.00%)
 - Country: 0 (0.00%)
 - Institution Name: 0 (0.00%)
Columns with >10% missing:
 - none
Date of Birth NaT after parsing attempts: 0
Exact duplicate rows remaining: 0

--- summary for cp ---
Records: 6839 rows x 16 cols
Top 10 columns by missing values:
 - Learner SignUp DateTime: 0 (0.00%)
 - Opportunity Id: 0 (0.00%)
 - Opportunity Name: 0 (0.00%)
 - Opportunity Category: 0 (0.00%)
 - Opportunity End Date: 0 (0.00%)
 - First Name: 0 (0.00%)
 - Date of Birth: 0 (0.00%)
 - Gender: 0 (0.00%)
 - Country: 0 (0.00%)
 - Institution Name: 0 (0.00%)
Columns with >10% missing:
 - none
Date of Birth NaT after parsing attempts: 0
Exact duplicate rows remaining: 0

--